In [1]:
import numpy as np
import matplotlib.pyplot as plt
from hardware.waveform import Idle, Sin, IQWaveform, Sequence
from hardware.awg5000 import AWG5014

awg = AWG5014(
    gpib='GPIB0::1::INSTR',
    ftp='169.254.103.111',
    socket=('169.254.103.111',4001),
)

sampling_rate = awg.get_sampling()
Hz,  sec    = 1./sampling_rate, sampling_rate
kHz, ms     = 1.e3*Hz,          1.e-3*sec
MHz, us     = 1.e6*Hz,          1.e-6*sec
GHz, ns     = 1.e9*Hz,          1.e-9*sec

In [ ]:
zero = Idle(5*ns)

sin = Sin(0, freq=100*kHz, amp=1.0, phase=0)


seqName = "rabi_test"
seq=Sequence(seqName)
main_wave = seq.fname
waves_list = []

tau_start   = 50*us
tau_delta   = 50*us
n_inc       = 3
tau_end     = tau_start + n_inc*tau_delta
tau_list = np.arange(tau_start, tau_end + tau_delta/2, tau_delta)
for i, t in enumerate(tau_list):
    name = 'RABI_%03d' % i
    sin.duration = t
    pulse_sequence = [zero, sin]
    wfm = IQWaveform(name, pulse_sequence, sampling=sampling_rate, file_type='WFM')
    waves_list.append(wfm[0])
    waves_list.append(wfm[1])
    seq.append(wfm, wait=True)

waves_list.append(seq)
awg.ftp_cwd = '/rabi'
awg.upload(waves_list)
awg.managed_load(seq.fname, cwd='/rabi')
# %matplotlib qt
# waves.i.plot(t_scale=ns, show=False)
# waves.q.plot(t_scale=ns, color='tab:blue')

/rabi


Success! <class 'hardware.waveform.Waveform'>
Success! <class 'hardware.waveform.Waveform'>
Success! <class 'hardware.waveform.Waveform'>
Success! <class 'hardware.waveform.Waveform'>
Success! <class 'hardware.waveform.Waveform'>
Success! <class 'hardware.waveform.Waveform'>
Success! <class 'hardware.waveform.Waveform'>
Success! <class 'hardware.waveform.Waveform'>
Success! <class 'hardware.waveform.Sequence'>


In [2]:
def format_bytes(size):
    # 2**10 = 1024
    power = 2**10
    n = 0
    power_labels = {0 : '', 1: 'kilo', 2: 'mega', 3: 'giga', 4: 'tera'}
    while size > power:
        size /= power
        n += 1
    return size, power_labels[n]+'bytes'

'%d %s' % format_bytes(123123123)

'117 megabytes'

In [61]:
from numpy import sqrt
from numpy.random import random

from traits.api import HasTraits, Property, Array, Float
from traitsui.api import View, Item, TabularAdapter, TabularEditor


# -- Tabular Adapter Definition -------------------------------------------
class ArrayAdapter(TabularAdapter):

    columns = [('Index', 'index'), ('File name', 0), ('Type', 1), ('File size', 2)]

    font = 'Default 10'# 'Times New Roman 10'
    alignment = 'center'
    width = Float(100)
    #format = '%.4f'

    index_text = Property()

    def _get_index_text(self):
        return str(self.row)

    def get_width(self, trait, name, column):
        print(name, trait.__dict__, column)
        print(self.width)
        if column == 0:
            return 50
        elif column == 1:
            return 150
        elif column == 2:
            return 100
        elif column == 3:
            return 50


# -- ShowArray Class Definition -------------------------------------------


class ShowArray(HasTraits):

    data = Array

    traits_view = View(
        Item(
            'data',
            show_label=False,
            editor=TabularEditor(
                adapter=ArrayAdapter(),
                #auto_resize=True,
                # Do not allow any kind of editing of the array:
                editable=False,
                operations=[],
                drag_move=False,
            ),
        ),
        title='Array Viewer',
        width=600,
        height=800,
        resizable=True,
    )


# Create the demo:
demo = ShowArray(data=[['1','2',3], ['4','5',6]])
demo.configure_traits()

data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')} 0
100
data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')} 1
100
data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')} 2
100
data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')} 3
100
data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')} 0
100
data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')} 0
100
data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')} 1
100
data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')} 1
100
data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')} 2
100
data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')} 2
100
data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')} 3
100
data {'data': array([['1', '2', '3'],
       ['4', '5', '6']], dtype='<U11')

True

In [10]:
awg.managed_load(seq.fname, cwd='/rabi')

/rabi


In [5]:
awg.tell('*CLS')

In [1]:
awg.ask('SOUR1:FUNC:USER?')

NameError: name 'awg' is not defined

In [ ]:
awg.tell('SOUR1:FUNC:USER "/waves/test.SEQ"')

In [7]:
# awg.ftp_manager.load(seq.fname)
awg.ftp_manager.load_file

In [ ]:
awg.managed_load(seq.fname, cwd='/')

/


Exception in thread Thread-283:
Traceback (most recent call last):
  File "c:\Users\yy3\anaconda3\envs\pi3\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\yy3\pi3-mini\hardware\awg5000.py", line 218, in run
    raise e
  File "c:\Users\yy3\pi3-mini\hardware\awg5000.py", line 203, in run
    self.setup_ftp()
  File "c:\Users\yy3\pi3-mini\hardware\awg5000.py", line 199, in setup_ftp
    self.ftp.cwd(self.awg.ftp_cwd)
  File "c:\Users\yy3\anaconda3\envs\pi3\Lib\ftplib.py", line 625, in cwd
    return self.voidcmd(cmd)
           ^^^^^^^^^^^^^^^^^
  File "c:\Users\yy3\anaconda3\envs\pi3\Lib\ftplib.py", line 286, in voidcmd
    return self.voidresp()
           ^^^^^^^^^^^^^^^
  File "c:\Users\yy3\anaconda3\envs\pi3\Lib\ftplib.py", line 259, in voidresp
    resp = self.getresp()
           ^^^^^^^^^^^^^^
  File "c:\Users\yy3\anaconda3\envs\pi3\Lib\ftplib.py", line 254, in getresp
    raise error_perm(resp)
ftplib.error_perm: 550 Cannot create a file when th